# Modelo CNN con 32% de los datos

## 1. Librerias

In [ ]:
import os
import json
import gc

import numpy as np
import pandas as pd
import h5py

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

import boto3

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

## 2. Configuración global

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

tf.config.optimizer.set_jit(True)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

BUCKET = "tfm-mu-bd"
S3_BASE = f"s3://{BUCKET}/processed"
S3_OUT_KEY = "outputs/cnn_lstm_binary"

BATCH_SIZE = 64
EPOCHS = 50

N_TRAIN = 8000
N_VAL = 2000
N_TEST = 2000

s3_client = boto3.client("s3")

print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"Train: {N_TRAIN} | Val: {N_VAL} | Test: {N_TEST} | Batch: {BATCH_SIZE}")

## 3. Modelo y entrenamiento

In [ ]:
def download_hdf5(split_name):
    local = f"/content/dl_{split_name}.h5"
    if os.path.exists(local):
        print(f" {local} ya existe ({os.path.getsize(local)/1e6:.0f} MB)")
        return local
    key = f"processed/deep_learning/dl_{split_name}.h5"
    print(f"Descargando {key}")
    s3_client.download_file(BUCKET, key, local)
    print(f" {os.path.getsize(local)/1e6:.0f} MB")
    return local

def load_split(hdf5_path, parquet_path, n_samples=None):
    import s3fs
    fs = s3fs.S3FileSystem()
    df = pd.read_parquet(parquet_path, filesystem=fs)
    label_map = dict(zip(df["record_id"].values,
                         df["is_adverse"].astype(int).values))
    del df; gc.collect()

    with h5py.File(hdf5_path, "r") as hf:
        record_ids = np.array([
            r.decode() if isinstance(r, bytes) else r
            for r in hf["record_id"][:]
        ])
        labels = np.array([label_map.get(rid, -1) for rid in record_ids])
        valid  = np.where(labels >= 0)[0]

        if n_samples and n_samples < len(valid):
            idx, _ = train_test_split(
                np.arange(len(valid)),
                train_size=n_samples,
                stratify=labels[valid],
                random_state=RANDOM_STATE
            )
            valid = valid[idx]

        labels = labels[valid]

        sort_order = np.argsort(valid)
        sorted_valid = valid[sort_order]

        print(f"Leyendo {len(sorted_valid)} señales")
        signals = hf["signal"][sorted_valid.tolist()]
        signals = signals.transpose(0, 2, 1).astype(np.float32)

        restore = np.argsort(sort_order)
        signals = signals[restore]

    print(f" {signals.shape} | {signals.nbytes/1e9:.2f} GB | "
          f"Adversos: {labels.sum()} ({labels.mean()*100:.1f}%)")
    return signals, labels

def make_dataset(signals, labels, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (signals, labels.astype(np.float32))
    )
    if shuffle:
        ds = ds.shuffle(len(labels), seed=RANDOM_STATE)
    ds = ds.batch(batch_size)
    ds = ds.cache()
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
def build_model():
    inp = keras.Input(shape=(5000, 12), name="signal")

    x = layers.Conv1D(32, 7, padding="same",
                      kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(64, 5, padding="same",
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 5, padding="same",
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 3, padding="same",
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Bidirectional(
        layers.LSTM(64, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inputs=inp, outputs=out, name="CNN_LSTM")

def train(model, ds_train, ds_val, class_weight):
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )
    cb = [
        callbacks.EarlyStopping(
            monitor="val_auc", patience=10,
            restore_best_weights=True, mode="max", verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_auc", factor=0.5, patience=5,
            mode="max", min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            "/content/best_cnn_lstm.keras",
            monitor="val_auc", save_best_only=True,
            mode="max", verbose=0
        )
    ]
    return model.fit(
        ds_train, validation_data=ds_val,
        epochs=EPOCHS, class_weight=class_weight,
        callbacks=cb, verbose=1
    )

## 4. Evaluación y resultados

In [ ]:
def evaluate(model, ds, y_true):
    y_prob = model.predict(ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    f1  = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n  AUC-ROC: {auc:.4f}")
    print(f" AUC-PR: {ap:.4f}")
    print(f" F1: {f1:.4f}")
    print(f" Precision: {prec:.4f}")
    print(f" Recall: {rec:.4f}")
    print(f" Spec: {tn/(tn+fp):.4f}")
    print(f" CM:\n{cm}")

    return {"auc_roc": round(auc,4), "auc_pr": round(ap,4),
            "f1": round(f1,4), "precision": round(prec,4),
            "recall": round(rec,4), "specificity": round(tn/(tn+fp),4),
            "cm": cm.tolist(), "tp":int(tp),"tn":int(tn),
            "fp":int(fp),"fn":int(fn)}, y_prob


In [ ]:
def upload_s3(results):
    with open("/content/cnn_lstm_results.json","w") as f:
        json.dump(results, f, indent=2)
    for local in [
        "/content/cnn_lstm_learning_curves.png",
        "/content/cnn_lstm_roc_pr.png",
        "/content/cnn_lstm_confusion_matrix.png",
        "/content/best_cnn_lstm.keras",
        "/content/cnn_lstm_results.json",
    ]:
        key = f"{S3_OUT_KEY}/{os.path.basename(local)}"
        try:
            s3_client.upload_file(local, BUCKET, key)
            print(f"{os.path.basename(local)}")
        except Exception as e:
            print(f"{os.path.basename(local)}: {e}")

## 5. Ejecución principal

In [ ]:
if __name__ == "__main__":

    print("\n Descarga HDF5")
    path_tr = download_hdf5("train")
    path_vl = download_hdf5("val")
    path_te = download_hdf5("test")

    print("\n Train")
    sig_tr, lbl_tr = load_split(
        path_tr, f"{S3_BASE}/machine_learning/ml_train.parquet", N_TRAIN)
    neg, pos = (lbl_tr==0).sum(), (lbl_tr==1).sum()
    cw = {0: (neg+pos)/(2*neg), 1: (neg+pos)/(2*pos)}
    print(f"  class_weight: {cw}")
    ds_train = make_dataset(sig_tr, lbl_tr, BATCH_SIZE, shuffle=True)
    del sig_tr, lbl_tr; gc.collect()
    print("  RAM liberada")

    print("\n>>> Val")
    sig_vl, lbl_vl = load_split(
        path_vl, f"{S3_BASE}/machine_learning/ml_val.parquet", N_VAL)
    ds_val = make_dataset(sig_vl, lbl_vl, BATCH_SIZE, shuffle=False)
    del sig_vl; gc.collect()
    print("  RAM liberada")

    print("\n>>> Test")
    sig_te, lbl_te = load_split(
        path_te, f"{S3_BASE}/machine_learning/ml_test.parquet", N_TEST)
    ds_test     = make_dataset(sig_te, lbl_te, BATCH_SIZE, shuffle=False)
    lbl_te_eval = lbl_te.copy()
    del sig_te, lbl_te; gc.collect()
    print("  RAM liberada")

    print("\n>>>  Modelo")
    model = build_model()
    model.summary()

    print("\n>>> Entrenamiento")
    history = train(model, ds_train, ds_val, cw)

    print("\n>>> Evaluación (TEST)")
    r_test, y_prob = evaluate(model, ds_test, lbl_te_eval)


    print("\n COMPLETADO ")
    print(f"AUC-ROC: {r_test['auc_roc']} | F1: {r_test['f1']} | Recall: {r_test['recall']}")

## 6. Visualizaciones

In [ ]:
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(lbl_te_eval, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
            xticklabels=["No adverso", "Adverso"],
            yticklabels=["No adverso", "Adverso"])

plt.title("Confusion Matrix")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(lbl_te_eval, y_prob)
auc_score = roc_auc_score(lbl_te_eval, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color="#1d3557", label=f"AUC = {auc_score:.4f}")
plt.plot([0,1],[0,1],"--")

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(lbl_te_eval, y_prob)
ap = average_precision_score(lbl_te_eval, y_prob)

plt.figure(figsize=(6,5))
plt.plot(recall, precision, color="#1d3557", label=f"AP = {ap:.4f}")

plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
C1 = "#1d3557"
C2 = "#457b9d"

plt.figure(figsize=(6,5))

plt.plot(history.history["auc"], label="train AUC", color=C1, linewidth=2)
plt.plot(history.history["val_auc"], label="val AUC", color=C2, linewidth=2)

plt.title("Learning Curve (AUC)")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.tight_layout()
plt.show()